# WWW::HorizonsEphemerisSystem basic usage

Anton Antonov  
April 2026

---

## Introduction

This notebook has examples for the basic usage of the sub `horizons-ephemeris-data` provided by the package ["WWW::HorizonsEphemerisSystem"](https://raku.land/zef:antononcube/WWW::HorizonsEphemerisSystem).

---

## Setup

In [22]:
use WWW::HorizonsEphemerisSystem;
use Data::Summarizers;
use Data::TypeSystem;

----

## Basic usage

Cartesian position of Mars with default parameters.

In [23]:
my $mars-default = horizons-ephemeris-data(
    'state',
    '499',
    'position',
    :modifier<data>,
);

[{Calendar Date (TDB) => A.D. 2026-Nov-27 14:20:50.0000, Date => 2461372.097800926, Vx => -21.39541994831129, Vy => -7.358060313281046, Vz => 0.370319146027112, X => -94385686.07268773, Y => 222529607.4118334, Z => 7003075.142028347}]

Cartesian position of Mars for a specific date.

In [24]:
my $date = "{.year}-{.month.fmt('%02s')}-{.day.fmt('%02s')}" with DateTime.now(:12hours);

my $mars-at-date = horizons-ephemeris-data(
    'state',
    ['499', {
        dates => $date,
    }],
    'position',
    :modifier<data>,
);

[{Calendar Date (TDB) => A.D. 2026-Nov-27 00:00:00.0000, Date => 2461371.500000000, Vx => -21.44049181153535, Vy => -7.250543072849926, Vz => 0.3736780107277102, X => -93279446.55129047, Y => 222906875.8378729, Z => 6983861.389652953}]

Azimuth/Elevation of the Moon from Moscow (lat, lon) for next 12 hours.

In [25]:
my @dates = do with DateTime.now { ($_, $_.later(:12hours), $_.later(minutes => 12*60 + 10))};
@dates = @dates.map({ "{.year}-{.month.fmt('%02s')}-{.day.fmt('%02s')} {.hh-mm-ss}" });
say (:@dates);

my $tycho-from-moscow = horizons-ephemeris-data(
    'observer',
    {
        target => '301',
        center => (55.7505, 37.6175),
        :@dates,
    },
    <azimuth elevation>,
    :modifier<data>,
);

dates => [2026-11-27 14:20:51 2026-11-28 02:20:51 2026-11-28 02:30:51]


[{Azimuth => 27.270259, Elevation => -5.899567, ObservationDateUT => 2026-11-27T14:20:51Z} {Azimuth => 217.111351, Elevation => 53.146854, ObservationDateUT => 2026-11-28T02:20:51Z} {Azimuth => 220.490271, Elevation => 52.264391, ObservationDateUT => 2026-11-28T02:30:50.999962Z}]

Here is a range of dates from now to 4 hours from now with step 30 min:

In [26]:
my $start = DateTime.now;
my $end = $start.later(:4hours);
my $current = $start;
my @dates;

while $current <= $end {
    @dates.push($current);
    $current = $current.later(:30minutes);
}

#@dates = @dates».Str.map({ $_.substr(^19).subst('T', ' ') });
@dates = @dates.map({ "{.year}-{.month.fmt('%02s')}-{.day.fmt('%02s')} {.hh-mm-ss}" });


deduce-type(@dates);

Vector(Atom((Str)), 9)

Orbital elements of Saturn moon Pandora for a specific list of date-times.

In [27]:
my $pandora-elements = horizons-ephemeris-data(
    'orbital-elements',
    ['617', {
        center => '500@699',
        :@dates
    }],
    'all',
    :modifier<association>,
);

deduce-type($pandora-elements)

Assoc(Atom((Str)), Struct([ApoapsisDistance, AscendingNodeLongitude, Calendar Date (TDB), Date, Eccentricity, Inclination, MeanAnomaly, MeanMotion, OrbitalPeriod, PeriapsisDate, PeriapsisDistance, PerifocusArgument, SemiMajorAxis, TrueAnomaly], [Num, Num, Str, Str, Num, Num, Num, Num, Num, Str, Num, Num, Num, Num]), 9)

In [28]:
#%html
$pandora-elements.values
==> to-html(field-names => ['Calendar Date (TDB)', 'Date', |$pandora-elements.values.head.keys.grep({ $_ ∉ ['Calendar Date (TDB)', 'Date'] }).sort])

Calendar Date (TDB),Date,ApoapsisDistance,AscendingNodeLongitude,Eccentricity,Inclination,MeanAnomaly,MeanMotion,OrbitalPeriod,PeriapsisDate,PeriapsisDistance,PerifocusArgument,SemiMajorAxis,TrueAnomaly
A.D. 2026-Nov-27 16:50:51.0000,2461372.201979167,143319.0316678895,169.4378605092672,0.006794145215061469,28.02187240227712,36.62001230992847,0.006570163594970106,54793.15618192578,2.461372137468942E+06,141384.7130827817,201.8927903439695,142351.8723753356,37.08760368799351
A.D. 2026-Nov-27 17:50:51.0000,2461372.243645833,143136.6445734287,169.4379722114127,0.005528634030189289,28.02188781965895,47.76671711255037,0.006570317769525138,54791.87044343192,2.461372159501422E+06,141562.6463850649,214.6123633492865,142349.6454792468,48.23797866783114
A.D. 2026-Nov-27 17:20:51.0000,2461372.222812500,143232.4456610709,169.4378957891603,0.006193730708147273,28.02187977727997,42.22241366249455,0.006570240374142109,54792.51587458183,2.461372148433890E+06,141469.0810722861,208.2236281166998,142350.7633666785,42.70211661376015
A.D. 2026-Nov-27 15:20:51.0000,2461372.139479167,143513.4640061786,169.4379321662176,0.008140010821906568,28.02192479212111,19.65818662420274,0.006569968167988653,54794.78603169721,2.461372104848054E+06,141195.92648647,183.0538052560581,142354.6952463243,19.97501325074977
A.D. 2026-Nov-27 16:20:51.0000,2461372.181145833,143395.3473567563,169.437862530983,0.007322792048626284,28.02187354951194,30.98320406700915,0.006570090618405742,54793.76479092694,2.461372126564866E+06,141310.5056005234,195.5961635136535,142352.9264786399,31.41858859237495
A.D. 2026-Nov-27 14:20:51.0000,2461372.097812500,143580.7196651442,169.4379800350878,0.008604795296696215,28.02205108678121,8.306911707882078,0.006569893257231012,54795.41080880937,2.461372083178349E+06,141130.8350184874,170.5372669018568,142355.7773418158,8.450901644375401
A.D. 2026-Nov-27 15:50:51.0000,2461372.160312500,143460.4363172194,169.4378914801884,0.00777327273189482,28.02188969804066,25.32634068613402,0.006570024552562692,54794.31577764485,2.461372115696389E+06,141247.3252377653,189.319413530274,142353.8807774924,25.71075261488254
A.D. 2026-Nov-27 18:20:51.0000,2461372.264479167,143032.7729306097,169.4380865307027,0.004806539339180077,28.02188866593905,53.21187695960489,0.006570392304065181,54791.24888437234,2.461372170743789E+06,141664.3649376938,221.1000506223484,142348.5689341518,53.65456744047801
A.D. 2026-Nov-27 14:50:51.0000,2461372.118645833,143553.7368666965,169.437967195809,0.008418375662540825,28.02197950700686,13.98402315096083,0.006569923798329227,54795.15608560792,2.461372094010522E+06,141156.9354718163,176.7940956735406,142355.3361692564,14.21954133483528


Using date range spec (the third element is automatically recognized to be a time step):

In [29]:
my $start = DateTime.now;
my $end = $start.later(:12hours);
my @date-range = $start, $end, "10 m";
@date-range .= map({ $_ ~~ DateTime ?? "{.year}-{.month.fmt('%02s')}-{.day.fmt('%02s')} {.hh-mm-ss}" !! $_ });
say (:@date-range);

my $pandora-elements-range = horizons-ephemeris-data(
    'orbital-elements',
    ['617', {
        center => '699',
        dates => @date-range
    }],
    'all',
    :modifier<association>,
);

deduce-type($pandora-elements)

date-range => [2026-11-27 14:20:52 2026-11-28 02:20:52 10 m]


Assoc(Atom((Str)), Struct([ApoapsisDistance, AscendingNodeLongitude, Calendar Date (TDB), Date, Eccentricity, Inclination, MeanAnomaly, MeanMotion, OrbitalPeriod, PeriapsisDate, PeriapsisDistance, PerifocusArgument, SemiMajorAxis, TrueAnomaly], [Num, Num, Str, Str, Num, Num, Num, Num, Num, Str, Num, Num, Num, Num]), 9)

Show a sample:

In [30]:
#%html
$pandora-elements-range.values
==> { .pick(6).sort(*<Date>)}()
==> to-html(field-names => ['Calendar Date (TDB)', 'Date', |$pandora-elements-range.values.head.keys.grep({ $_ ∉ ['Calendar Date (TDB)', 'Date'] }).sort])

Calendar Date (TDB),Date,ApoapsisDistance,AscendingNodeLongitude,Eccentricity,Inclination,MeanAnomaly,MeanMotion,OrbitalPeriod,PeriapsisDate,PeriapsisDistance,PerifocusArgument,SemiMajorAxis,TrueAnomaly
A.D. 2026-Nov-27 14:30:52.0000,2461372.104768518,143573.21438244,169.4379791090586,0.008552950495633876,28.02202561333822,10.20263944963517,0.006569901828681985,54795.33931973852,2.461372086794718E+06,141138.092667693,172.6261538740088,142355.6535250665,10.37808896260244
A.D. 2026-Nov-27 15:00:52.0000,2461372.125601852,143541.7450818084,169.4379572018637,0.008335503699377937,28.02195915836055,15.87900709278609,0.006569937153052841,54795.04470339102,2.461372097628250E+06,141168.5414352941,178.8837090359332,142355.1432585512,16.14299403140255
A.D. 2026-Nov-27 15:20:52.0000,2461372.139490741,143513.4380301943,169.4379321435558,0.008139831231278895,28.02192476707815,19.66133763512758,0.006569968196171535,54794.78579664661,2.461372104854077E+06,141195.9516482514,183.0572841476942,142354.6948392229,19.9782059174749
A.D. 2026-Nov-27 19:20:52.0000,2461372.306157407,142805.6539584619,169.4383856353466,0.00322407018729461,28.02184501735586,63.42425382924859,0.00657052044642057,54790.18031153341,2.461372194434597E+06,141887.78233798,234.7595161774969,142346.7181482209,63.7552651336191
A.D. 2026-Nov-27 22:40:52.0000,2461372.445046296,142720.3963615079,169.4388395653691,0.002627868271046798,28.02137814735514,293.4743589439843,0.006570547393058943,54789.95561013687,2.461372562231768E+06,141972.2615587593,84.25879478028588,142346.3289601336,293.1977887452126
A.D. 2026-Nov-28 01:50:52.0000,2461372.576990741,143367.6068442942,169.4389746500355,0.007129372656771259,28.02132586548307,326.8942328299638,0.006570104826760717,54793.64629521324,2.461372635310754E+06,141337.8356473602,126.4138864901944,142352.7212458272,326.4446668249335


Here is summary for some of the columns:

In [31]:
my @field-names = <Date Eccentricity Inclination MeanMotion>;
sink records-summary($pandora-elements-range.values.map({ $_.grep(*.key ∈ @field-names) })».Hash, :@field-names)

+-------------------------+---------------------------------+------------------------------+--------------------------------+
| Date                    | Eccentricity                    | Inclination                  | MeanMotion                     |
+-------------------------+---------------------------------+------------------------------+--------------------------------+
| 2461372.472824074 => 1  | Min    => 0.0002676379378662241 | Min    => 28.02130619144841  | Min    => 0.006569893270104637 |
| 2461372.174212963 => 1  | 1st-Qu => 0.0026467506004783214 | 1st-Qu => 28.02132612908286  | 1st-Qu => 0.006570121359148083 |
| 2461372.195046296 => 1  | Mean   => 0.004819449506171018  | Mean   => 28.02164668521195  | Mean   => 0.006570326078693546 |
| 2461372.306157407 => 1  | Median => 0.005052672221306841  | Median => 28.0217288688179   | Median => 0.006570355366284492 |
| 2461372.556157407 => 1  | 3rd-Qu => 0.007053869171604121  | 3rd-Qu => 28.021882119360647 | 3rd-Qu => 0.0065705505951